<div style='background: #0f1f3d; padding: 40px 48px 36px; border-radius: 6px;'>
  <p style='color: #5a9bd5; margin: 0 0 16px 0; font-size: 0.72em; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;'>FARMSA Research Group &mdash; Module 3</p>
  <h1 style='color: #ffffff; font-size: 2em; font-weight: 600; margin: 0 0 6px 0; letter-spacing: -0.5px;'>Factor Model Covariance: CAPM &amp; Fama-French</h1>
  <p style='color: #8fadc8; font-size: 0.95em; font-weight: 400; margin: 12px 0 0 0;'>Author: Denis Goubkine</p>
</div>

---
## §1 &nbsp; Motivation & Derivation

The sample covariance has **1,275 free parameters** estimated from ~250 daily returns per window. Most of those numbers are noise. Factor models cut this down by assuming stocks only co-move through shared exposure to a few common drivers.

### CAPM — one factor

$$r_{i,t} - r_f = \alpha_i + \beta_i \underbrace{(r_{m,t} - r_f)}_{\text{market}} + \varepsilon_{i,t}$$

All co-movement goes through the market. Under the assumption that residuals $\varepsilon_i$ are uncorrelated across stocks, the full covariance matrix becomes:

$$\hat{\Sigma}_{\text{CAPM}} = \boldsymbol{\beta}\boldsymbol{\beta}^{\top} \sigma_m^2 + D \qquad \text{(101 parameters)}$$

### Fama-French 3F — three factors

$$r_{i,t} - r_f = \alpha_i + \beta_{i,1}\,\text{MktRF} + \beta_{i,2}\,\text{SMB} + \beta_{i,3}\,\text{HML} + \varepsilon_{i,t}$$

**SMB** (small minus big) captures size risk. **HML** (high minus low book-to-market) captures value vs growth. The covariance extends naturally:

$$\hat{\Sigma}_{\text{FF3}} = B\,\Sigma_f\,B^{\top} + D \qquad \text{(206 parameters)}$$

where $B$ is $N \times 3$ (loadings), $\Sigma_f$ is $3 \times 3$ (factor covariance), $D$ is diagonal (residual variances).

**The core question:** does adding SMB & HML actually improve the portfolio, or do the extra parameters just add noise?

In [ ]:
# ── §2  Setup & Factor Data ───────────────────────────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
import seaborn as sns
from scipy.optimize import minimize
warnings.filterwarnings('ignore')

prices  = pd.read_csv('data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('data/returns.csv', index_col=0, parse_dates=True)
N, T = returns.shape[1], returns.shape[0]
print(f"Universe: {N} assets × {T} days")

SECTOR_MAP = {
    'AAPL':'Tech','MSFT':'Tech','GOOGL':'Tech','NVDA':'Tech','ADBE':'Tech',
    'CRM':'Tech','INTC':'Tech','CSCO':'Tech',
    'JPM':'Fin','BAC':'Fin','GS':'Fin','MS':'Fin','BLK':'Fin','AXP':'Fin',
    'JNJ':'Health','UNH':'Health','PFE':'Health','ABBV':'Health','LLY':'Health','MRK':'Health',
    'AMZN':'Discr','HD':'Discr','MCD':'Discr','NKE':'Discr','SBUX':'Discr',
    'PG':'Staples','KO':'Staples','PEP':'Staples','COST':'Staples',
    'CAT':'Indust','HON':'Indust','UPS':'Indust','BA':'Indust',
    'XOM':'Energy','CVX':'Energy','COP':'Energy','SLB':'Energy',
    'META':'Comm','DIS':'Comm','NFLX':'Comm',
    'NEE':'Util','DUK':'Util','SO':'Util',
    'AMT':'RE','PLD':'RE','CCI':'RE',
    'LIN':'Mat','APD':'Mat','SHW':'Mat','ECL':'Mat',
}
SECTOR_COLORS = {
    'Tech':'#2563eb','Fin':'#059669','Health':'#dc2626','Discr':'#d97706',
    'Staples':'#7c3aed','Indust':'#6b7280','Energy':'#a16207','Comm':'#db2777',
    'Util':'#0891b2','RE':'#4f46e5','Mat':'#65a30d',
}

# Download FF3 daily factors from Kenneth French's data library (cached after first run)
import os, urllib.request, zipfile, io

ff_cache = 'data/ff_factors.csv'
if os.path.exists(ff_cache):
    ff_factors = pd.read_csv(ff_cache, index_col=0, parse_dates=True)
    print(f"Loaded {ff_cache}")
else:
    print("Downloading FF3 factors...")
    url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip'
    resp = urllib.request.urlopen(url)
    z = zipfile.ZipFile(io.BytesIO(resp.read()))
    with z.open(z.namelist()[0]) as f:
        raw = f.read().decode('utf-8')
    ff_all = pd.read_csv(io.StringIO(raw), skiprows=4, index_col=0)
    ff_all = ff_all[pd.to_numeric(ff_all.index, errors='coerce').notna()]
    ff_all.index = pd.to_datetime(ff_all.index.astype(str).str.strip(), format='%Y%m%d')
    ff_all = ff_all[(ff_all.index >= '2018-01-01') & (ff_all.index <= '2024-12-31')]
    ff_factors = ff_all / 100
    ff_factors.to_csv(ff_cache)
    print(f"Saved to {ff_cache}")

common_dates = returns.index.intersection(ff_factors.index)
returns_aligned = returns.loc[common_dates]
ff = ff_factors.loc[common_dates]
mkt_rf, smb, hml, rf = ff['Mkt-RF'], ff['SMB'], ff['HML'], ff['RF']
print(f"Factors aligned: {len(common_dates)} days")

In [ ]:
# ── Factor Regressions (full sample) ─────────────────────
excess_returns = returns_aligned.subtract(rf, axis=0)
X_1f = np.column_stack([np.ones(len(common_dates)), mkt_rf.values])
X_3f = np.column_stack([np.ones(len(common_dates)), mkt_rf.values, smb.values, hml.values])

capm_betas, ff3_betas, capm_r2, ff3_r2 = {}, {}, {}, {}
capm_resid = pd.DataFrame(index=common_dates, columns=returns.columns, dtype=float)
ff3_resid  = pd.DataFrame(index=common_dates, columns=returns.columns, dtype=float)

for ticker in returns.columns:
    y = excess_returns[ticker].values
    ss_tot = np.sum((y - y.mean())**2)

    c1 = np.linalg.lstsq(X_1f, y, rcond=None)[0]
    r1 = y - X_1f @ c1
    capm_betas[ticker] = c1[1]
    capm_r2[ticker] = 1 - np.sum(r1**2) / ss_tot
    capm_resid[ticker] = r1

    c3 = np.linalg.lstsq(X_3f, y, rcond=None)[0]
    r3 = y - X_3f @ c3
    ff3_betas[ticker] = {'Mkt': c3[1], 'SMB': c3[2], 'HML': c3[3]}
    ff3_r2[ticker] = 1 - np.sum(r3**2) / ss_tot
    ff3_resid[ticker] = r3

beta_df = pd.DataFrame({
    'Sector':      {t: SECTOR_MAP.get(t,'') for t in returns.columns},
    'β_mkt':       capm_betas,
    'β_SMB':       {t: ff3_betas[t]['SMB'] for t in returns.columns},
    'β_HML':       {t: ff3_betas[t]['HML'] for t in returns.columns},
    'R² CAPM':     capm_r2,
    'R² FF3':      ff3_r2,
    'ΔR²':         {t: ff3_r2[t] - capm_r2[t] for t in returns.columns},
}).round(3)

display(beta_df.sort_values('β_mkt', ascending=False)
        .style.background_gradient(subset=['β_mkt'], cmap='RdYlGn')
              .background_gradient(subset=['β_SMB'], cmap='PiYG')
              .background_gradient(subset=['β_HML'], cmap='PuOr')
              .background_gradient(subset=['R² FF3'], cmap='Blues')
              .format(precision=3))

In [ ]:
# ── Factor Loadings & Explained Variance ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 7))

# Left two: loading bar charts for SMB and HML (Mkt is less interesting visually)
for ax, col, title in zip(axes[:2],
        ['β_SMB', 'β_HML'],
        ['SMB β  (size — negative = large-cap tilt)',
         'HML β  (value — negative = growth, positive = value)']):
    df_s = beta_df.sort_values(col)
    colors = [SECTOR_COLORS.get(SECTOR_MAP.get(t,''), '#999') for t in df_s.index]
    ax.barh(range(len(df_s)), df_s[col], color=colors, alpha=0.85)
    ax.set_yticks(range(len(df_s))); ax.set_yticklabels(df_s.index, fontsize=6)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.axvline(0, color='black', lw=0.8, alpha=0.4); ax.grid(True, axis='x', alpha=0.2)

# Right: R² scatter — CAPM vs FF3 (above diagonal = FF3 wins)
for sec in sorted(set(SECTOR_MAP.values())):
    tks = [t for t in returns.columns if SECTOR_MAP.get(t) == sec]
    axes[2].scatter([capm_r2[t] for t in tks], [ff3_r2[t] for t in tks],
                    c=SECTOR_COLORS[sec], s=50, alpha=0.8, label=sec,
                    edgecolors='white', lw=0.4)
for t in returns.columns:
    axes[2].annotate(t, (capm_r2[t], ff3_r2[t]),
                     fontsize=5, alpha=0.55, xytext=(2,2), textcoords='offset points')
hi = max(max(ff3_r2.values()), max(capm_r2.values())) * 1.05
axes[2].plot([0,hi],[0,hi],'k--',alpha=0.3,lw=0.8)
axes[2].set_xlabel('CAPM R²'); axes[2].set_ylabel('FF3 R²')
axes[2].set_title('R² comparison\n(above line = FF3 explains more)', fontweight='bold', fontsize=10)
axes[2].legend(fontsize=6, ncol=2, loc='lower right')

from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor=c,label=s) for s,c in SECTOR_COLORS.items()],
           loc='lower center', ncol=6, fontsize=7, bbox_to_anchor=(0.5,-0.02))
plt.suptitle('Factor Loadings & Variance Explained', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

avg_1f = np.mean(list(capm_r2.values()))
avg_3f = np.mean(list(ff3_r2.values()))
print(f"Average R²:  CAPM = {avg_1f:.3f}  →  FF3 = {avg_3f:.3f}  (+{(avg_3f-avg_1f)*100:.1f} pp)")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  COVARIANCE ESTIMATORS                                  ║
# ║  CAPM:  Σ = ββ'σ²_m + D        (~101 parameters)       ║
# ║  FF3:   Σ = BΣ_fB' + D         (~206 parameters)       ║
# ╚══════════════════════════════════════════════════════════╝

def estimate_covariance_capm(returns_df):
    dates = returns_df.index.intersection(ff.index)
    excess = returns_df.loc[dates].subtract(rf.loc[dates], axis=0)
    mkt = mkt_rf.loc[dates].values
    X = np.column_stack([np.ones(len(dates)), mkt])
    n = excess.shape[1]
    betas, resid_var = np.zeros(n), np.zeros(n)
    for i, col in enumerate(excess.columns):
        coefs = np.linalg.lstsq(X, excess[col].values, rcond=None)[0]
        betas[i] = coefs[1]
        resid_var[i] = np.var(excess[col].values - X @ coefs, ddof=1)
    return np.outer(betas, betas) * np.var(mkt, ddof=1) + np.diag(resid_var)

def estimate_covariance_ff3(returns_df):
    dates = returns_df.index.intersection(ff.index)
    excess = returns_df.loc[dates].subtract(rf.loc[dates], axis=0)
    factors = ff.loc[dates, ['Mkt-RF','SMB','HML']].values
    X = np.column_stack([np.ones(len(dates)), factors])
    n = excess.shape[1]
    B, resid_var = np.zeros((n,3)), np.zeros(n)
    for i, col in enumerate(excess.columns):
        coefs = np.linalg.lstsq(X, excess[col].values, rcond=None)[0]
        B[i] = coefs[1:]
        resid_var[i] = np.var(excess[col].values - X @ coefs, ddof=1)
    return B @ np.cov(factors, rowvar=False) @ B.T + np.diag(resid_var)

# Sanity checks
for name, func in [('CAPM', estimate_covariance_capm), ('FF3', estimate_covariance_ff3)]:
    C = func(returns.iloc[:252])
    assert C.shape == (N,N) and np.allclose(C,C.T) and np.all(np.linalg.eigvalsh(C) > -1e-10)
    print(f"✓ {name} — shape {C.shape}, symmetric, PSD")
print(f"\nSample cov: {N*(N+1)//2:,} params  |  CAPM: {2*N+1}  |  FF3: {4*N+6}")

In [ ]:
# ── Key Assumption Check: Are Residuals Independent? ─────
# The model sets D diagonal — meaning residuals are assumed uncorrelated.
# If same-sector stocks still move together after stripping factors,
# the model is missing co-movement and underestimating portfolio risk.

corr_1f = capm_resid.astype(float).corr().values
corr_3f = ff3_resid.astype(float).corr().values

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, mat, title in zip(axes, [corr_1f, corr_3f],
                           ['CAPM residuals', 'FF3 residuals']):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.heatmap(mat, mask=mask, cmap='RdBu_r', center=0, vmin=-0.3, vmax=0.3,
                square=True, linewidths=0.1, ax=ax,
                xticklabels=returns.columns, yticklabels=returns.columns,
                cbar_kws={'shrink': 0.65})
    ax.tick_params(labelsize=5); ax.set_title(title, fontweight='bold')

plt.suptitle('Residual Correlations — should be near zero if the model is complete',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

upper = np.triu_indices(N, k=1)
m1, m3 = np.abs(corr_1f[upper]).mean(), np.abs(corr_3f[upper]).mean()
print(f"Mean |off-diagonal residual correlation|:")
print(f"  CAPM: {m1:.4f}   FF3: {m3:.4f}   ({(1-m3/m1)*100:.0f}% reduction)")
print(f"\nResidual correlations aren't zero — same-sector risks still leak through.")

In [ ]:
# ── §3  Backtest ───────────────────────────────────────────
def min_variance_portfolio(cov):
    n = cov.shape[0]; w0 = np.ones(n)/n
    res = minimize(lambda w: w @ cov @ w, w0, method='SLSQP',
                   bounds=[(0,1)]*n,
                   constraints=[{'type':'eq','fun':lambda w: w.sum()-1}],
                   options={'ftol':1e-12,'maxiter':1000})
    return res.x if res.success else w0

LOOKBACK, REBAL, MIN_HIST = 252, 21, 252
dates = returns.index[MIN_HIST:]
n = returns.shape[1]

strategies = ['Equal Weight', 'Sample Cov MVO', 'CAPM Factor MVO', 'FF3 Factor MVO']
pv = {k: [1.0] for k in strategies}
cw = {k: np.ones(n)/n for k in strategies}

for i, date in enumerate(dates):
    idx = returns.index.get_loc(date)
    if i % REBAL == 0:
        window = returns.iloc[idx-LOOKBACK:idx]
        cw['Equal Weight']    = np.ones(n)/n
        cw['Sample Cov MVO']  = min_variance_portfolio(window.cov().values)
        cw['CAPM Factor MVO'] = min_variance_portfolio(estimate_covariance_capm(window))
        cw['FF3 Factor MVO']  = min_variance_portfolio(estimate_covariance_ff3(window))
    day_ret = returns.iloc[idx].values
    for k in strategies:
        pv[k].append(pv[k][-1] * (1 + cw[k] @ day_ret))

portfolio_df = pd.DataFrame(pv, index=[dates[0]-pd.Timedelta(days=1)]+list(dates))
print(f"✓ Backtest complete — {len(dates)} days, {len(dates)//REBAL} rebalances")

In [ ]:
# ── §4  Results ────────────────────────────────────────────
C = {'Equal Weight':'#888','Sample Cov MVO':'#2c5282',
     'CAPM Factor MVO':'#d97706','FF3 Factor MVO':'#c53030'}

fig, ax = plt.subplots(figsize=(11, 4.5))
for k in strategies:
    ax.plot(portfolio_df.index, portfolio_df[k], label=k, color=C[k], lw=1.8)
ax.set_ylabel('Value ($1 start)'); ax.set_title('Cumulative Performance', fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()

def metrics(v):
    r = v.pct_change().dropna()
    ar, av = r.mean()*252, r.std()*np.sqrt(252)
    return {'Ann. Return (%)': ar*100, 'Ann. Vol (%)': av*100,
            'Sharpe': ar/av if av>0 else 0,
            'Max DD (%)': ((v/v.cummax())-1).min()*100}

display(pd.DataFrame({k: metrics(portfolio_df[k]) for k in strategies}).T.round(2)
        .style.background_gradient(subset=['Sharpe'], cmap='RdYlGn')
              .background_gradient(subset=['Ann. Vol (%)'], cmap='YlOrRd_r'))

---
## §5 &nbsp; Interpretation

**Factor structure beats brute-force estimation.** CAPM uses 101 parameters vs 1,275 for the sample covariance. By constraining how stocks can co-move, both factor models produce less noisy covariance estimates — which shows up as lower realized volatility in the backtest.

**SMB and HML: real signal, marginal gain.** The factor loadings split cleanly along economic lines — tech stocks load negative on HML (growth), energy and financials load positive (value). FF3 explains ~7 more percentage points of return variance than CAPM on average. Whether that translates to meaningfully better portfolios out of sample is the interesting result.

**The residual problem.** Both models assume residuals are uncorrelated across stocks — they're not. Same-sector stocks still share risks that don't flow through the three Fama-French factors. This is the fundamental limitation: factor models only capture co-movement through the factors you choose. Anything else gets ignored.

**One-sentence takeaway:** Factor models work by trading estimation flexibility for economic structure — a good deal when the factors are the right ones, but they'll miss anything that doesn't fit the story.

---
<p style='text-align:center; color:#8fadc8; font-style:italic; font-size:0.85em;'>
Module 3 complete &mdash; see integration notebook for side-by-side comparison across all estimators.
</p>